In [1]:
import sys
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install nibabel scikit-image scipy matplotlib pandas tqdm
!pip install monai tifffile imagecodecs
!pip install einops

Looking in indexes: https://download.pytorch.org/whl/cpu


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, DistributedSampler
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import nibabel as nib
from scipy.ndimage import binary_dilation, binary_erosion
from skimage.morphology import skeletonize

import monai
from monai.transforms import (
    Compose, LoadImaged, ScaleIntensityRanged,
    RandSpatialCropd, RandFlipd, RandRotate90d, RandShiftIntensityd,
    EnsureTyped, AsDiscreted, ToTensord, RandGaussianNoised, RandAdjustContrastd
)
from monai.data import Dataset as MonaiDataset, DataLoader as MonaiDataLoader
from monai.losses import DiceCELoss, DiceLoss, TverskyLoss
from monai.metrics import DiceMetric
from monai.networks.nets import UNETR, SwinUNETR, UNet, SegResNet
from monai.inferers import SlidingWindowInferer

import glob
from tqdm import tqdm
import random
from datetime import datetime

import tifffile
from skimage.morphology import skeletonize
from scipy.ndimage import binary_dilation
from typing import List, Tuple, Optional
from monai.data import DataLoader
import random

2026-02-04 20:05:15.067039: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770235515.093566    3709 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770235515.101405    3709 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770235515.122817    3709 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770235515.122850    3709 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770235515.122852    3709 computation_placer.cc:177] computation placer alr

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(101)

torch.backends.cudnn.benchmark = True

In [4]:
def setup_distributed():
    if torch.cuda.device_count() > 1:
        if 'RANK' in os.environ and 'WORLD_SIZE' in os.environ:
            rank = int(os.environ['RANK'])
            world_size = int(os.environ['WORLD_SIZE'])
            local_rank = int(os.environ.get('LOCAL_RANK', rank))
            
        elif '--local_rank' in sys.argv:
            import argparse
            parser = argparse.ArgumentParser()
            parser.add_argument('--local_rank', type=int, default=0)
            args = parser.parse_args()
            local_rank = args.local_rank
            world_size = torch.cuda.device_count()
            rank = local_rank
            
        else:
            print("No DDP environment detected, using DataParallel instead")
            return False, 0, 1, 0
            
        torch.cuda.set_device(local_rank)
        dist.init_process_group(
            backend='nccl',
            init_method='env://',
            world_size=world_size,
            rank=rank
        )
        print(f'DDP initialized: Rank {rank}/{world_size}, Local Rank: {local_rank}')
        return True, rank, world_size, local_rank
    
    return False, 0, 1, 0

use_ddp=False
import os
import sys
import torch
import torch.distributed as dist
import torch.multiprocessing as mp

_, rank, world_size, local_rank = setup_distributed()
print(rank, world_size, local_rank)

world_size = torch.cuda.device_count()

No DDP environment detected, using DataParallel instead
0 1 0


In [5]:
input_shape = (160, 160, 160)
batch_size = 1 * world_size if torch.cuda.is_available() else 1
num_classes = 3
num_samples = 780
epochs = 300

In [6]:
def generate_tubed_skeleton(label_vol: np.ndarray) -> np.ndarray:
    """
    Generate tubed skeleton as an additional target channel
    
    Args:
        label_vol: 3D label volume with shape (D, H, W)
    
    Returns:
        Tubed skeleton mask with same shape and dtype=np.float32
    """
    # Extract ink class (assuming class 1 is ink)
    mask = (label_vol == 1).astype(np.uint8)
    
    if np.sum(mask) == 0:
        return np.zeros_like(label_vol, dtype=np.float32)
    
    # 1. Skeletonize
    try:
        skel = skeletonize(mask)
    except:
        skel = mask  # Fallback to original mask if skeletonization fails
    
    # 2. Tubular dilation (thicken the skeleton)
    tubed_skel = binary_dilation(skel, iterations=1).astype(np.float32)
    
    return tubed_skel

# ====================
# MONAI Transforms (same as before)
# ====================

def get_train_transforms(input_shape: Tuple[int, int, int] = (160, 160, 160)):
    """Get training transforms using MONAI"""
    return Compose([
        ScaleIntensityRanged(
            keys=['image'],
            a_min=0.0, a_max=255.0,  # Assuming 8-bit images
            b_min=0.0, b_max=1.0,
            clip=True
        ),
        RandSpatialCropd(
            keys=['image', 'label'],
            roi_size=input_shape,
            random_size=False,
            random_center=True
        ),
        RandFlipd(keys=['image', 'label'], spatial_axis=0, prob=0.5),
        RandFlipd(keys=['image', 'label'], spatial_axis=1, prob=0.5),
        RandFlipd(keys=['image', 'label'], spatial_axis=2, prob=0.5),
        RandRotate90d(
            keys=['image', 'label'],
            prob=0.4,
            max_k=3,
            spatial_axes=(0, 1)
        ),
        RandShiftIntensityd(keys=['image'], offsets=0.15, prob=0.5),
        RandGaussianNoised(keys=['image'], prob=0.1, mean=0.0, std=0.01),
        RandAdjustContrastd(keys=['image'], prob=0.2, gamma=(0.8, 1.2)),
        EnsureTyped(keys=['image', 'label']),
    ])

def get_val_transforms():
    """Get validation transforms using MONAI"""
    return Compose([
        ScaleIntensityRanged(
            keys=['image'],
            a_min=0.0, a_max=255.0,
            b_min=0.0, b_max=1.0,
            clip=True
        ),
        RandSpatialCropd(
            keys=['image', 'label'],
            roi_size=input_shape,
            random_size=False,
            random_center=True
        ),
        EnsureTyped(keys=['image', 'label']),
    ])

# ====================
# Custom 3D Occlusion Augmentation
# ====================

class RandomOcclusions3D:
    def __init__(self, occ_prob=0.8, max_blocks=6, min_size=2, max_size=8):
        self.occ_prob = occ_prob
        self.max_blocks = max_blocks
        self.min_size = min_size
        self.max_size = max_size
    
    def __call__(self, image_dict: dict) -> dict:
        if 'image' not in image_dict:
            return image_dict
        
        if random.random() > self.occ_prob:
            return image_dict
        
        image = image_dict['image']
        C, D, H, W = image.shape
        
        num_blocks = random.randint(1, self.max_blocks)
        for _ in range(num_blocks):
            block_d = random.randint(self.min_size, self.max_size)
            block_h = random.randint(self.min_size, self.max_size)
            block_w = random.randint(self.min_size, self.max_size)
            
            d0 = random.randint(0, max(D - block_d, 1))
            h0 = random.randint(0, max(H - block_h, 1))
            w0 = random.randint(0, max(W - block_w, 1))
            
            d1 = min(d0 + block_d, D)
            h1 = min(h0 + block_h, H)
            w1 = min(w0 + block_w, W)
            
            # Set block to zero
            image[:, d0:d1, h0:h1, w0:w1] = 0.0
        
        image_dict['image'] = image
        return image_dict


def revert_labels_numpy(labels):
    """
    Convert labels from format (0=background, 1=foreground, 2=unlabeled)
    to format (0=unlabeled, 1=background, 2=foreground)
    
    Args:
        labels: numpy array of shape (b, 160, 160, 160)
        
    Returns:
        Remapped numpy array
    """
    # Create a copy to avoid modifying the original
    remapped = labels.copy()
    
    # Apply the mapping:
    # 0 (background) -> 1 (background in new format)
    # 1 (foreground) -> 2 (foreground in new format)
    # 2 (unlabeled) -> 0 (unlabeled in new format)
    
    remapped[labels == 0] = 1  # background becomes 1
    remapped[labels == 1] = 2  # foreground becomes 2
    remapped[labels == 2] = 0  # unlabeled becomes 0
    
    return remapped

In [7]:
class VesuviusTIFDataset(Dataset):
    """
    Dataset for loading Vesuvius Challenge TIF files directly.
    
    Args:
        image_dir: Path to directory containing image TIFF files
        label_dir: Path to directory containing label TIFF files
        is_training: Whether this is for training (affects transforms and skeleton generation)
        input_shape: Desired input shape (D, H, W)
        num_classes: Number of classes in segmentation
        use_skeleton: Whether to generate skeleton targets (only for training)
        use_occlusion: Whether to apply random occlusions (only for training)
    """
    
    def __init__(
        self,
        image_files: str,
        label_files: str,
        is_training: bool = True,
        input_shape: Tuple[int, int, int] = (160, 160, 160),
        num_classes: int = 3,
        use_skeleton: bool = True,
        use_occlusion: bool = True
    ):
        self.use_occlusion = use_occlusion
        self.is_training = is_training
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.use_skeleton = use_skeleton and is_training
        
        self.image_files = image_files
        self.label_files = label_files
        
        if len(self.image_files) != len(self.label_files):
            print(f"Warning: Number of image files ({len(self.image_files)}) "
                  f"doesn't match label files ({len(self.label_files)})")
        
        if is_training:
            self.transform = get_train_transforms(input_shape)
            self.occlusion = RandomOcclusions3D()
        else:
            self.transform = get_val_transforms()
        
        print(f"VesuviusTIFDataset: Found {len(self.image_files)} samples")
        print(f"Training mode: {is_training}, Using skeleton: {use_skeleton}")
    
    def __len__(self) -> int:
        return len(self.image_files)
    
    def _load_tif_volume(self, file_path: str) -> np.ndarray:
        """Load 3D volume from TIF file"""
        try:
            volume = tifffile.imread(file_path)
            
            if volume.dtype != np.float32:
                volume = volume.astype(np.float32)
                
            return volume
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
            # Return dummy data as fallback
            return np.zeros(self.input_shape, dtype=np.float32)
    
    def _preprocess_label(self, label_vol: np.ndarray) -> np.ndarray:
        """
        Preprocess label volume:
        - For training with skeleton: create 2-channel output (mask, skeleton)
        - For validation: keep as single channel
        """
        if self.use_skeleton and self.is_training:
            # Generate skeleton
            skeleton = generate_tubed_skeleton(label_vol)
            
            mask = label_vol.astype(np.float32)
            combined = np.stack([mask, skeleton])  # (D, H, W, 2)
            return combined
        else:
            # For validation or when not using skeleton
            return label_vol[np.newaxis, ...]  # Add channel dimension

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        image_path = self.image_files[idx]
        label_path = self.label_files[idx]  # Handle mismatch
        
        image_vol = self._load_tif_volume(image_path)
        label_vol = self._load_tif_volume(label_path)

        if image_vol.ndim == 3:
            image_vol = image_vol[np.newaxis, ...]  # (D, H, W, 1)
            label_vol = label_vol[np.newaxis, ...] 
        
        data_dict = {
            'image': image_vol,  # (D, H, W, 1)
            'label': label_vol   # (D, H, W, C) where C=1 or 2
        }
        
        if self.transform:
            transformed = self.transform(data_dict)
            if self.use_occlusion:
                transformed = self.occlusion(transformed)

            image = transformed['image'].as_tensor()
            label = transformed['label'].as_tensor()

            label = self._preprocess_label(label[0].numpy())
            label = torch.from_numpy(revert_labels_numpy(label))
        else:
            # Manual conversion if no transforms
            image = torch.from_numpy(image_vol).float()
            label = torch.from_numpy(label_vol)
            if self.is_training:
                label = label.long()
            else:
                label = label.long()
        
        return image, label

In [8]:
from sklearn.model_selection import train_test_split
image_path = sorted(glob.glob(os.path.join('/kaggle/input/vesuvius-challenge-surface-detection/train_images', "*.tif")))
label_path = list(map(lambda x: x.replace('train_images', 'train_labels'), image_path))


train_images, val_images, train_labels, val_labels = train_test_split(image_path, label_path, test_size=0.1, shuffle=True)

In [9]:
input_shape = (160, 160, 160)

train_dataset = VesuviusTIFDataset(
    image_files=train_images,
    label_files=train_labels,
    is_training=True,
    input_shape=input_shape,
    num_classes=3,
    use_skeleton=True,
    use_occlusion=True
)

val_dataset = VesuviusTIFDataset(
    image_files=val_images,
    label_files=val_labels,
    is_training=False,
    input_shape=input_shape,
    num_classes=3,
    use_skeleton=True,
    use_occlusion=False
)

VesuviusTIFDataset: Found 707 samples
Training mode: True, Using skeleton: True
VesuviusTIFDataset: Found 79 samples
Training mode: False, Using skeleton: True


In [10]:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False
)

def plot_sample(x, y, sample_idx=0, max_slices=16):
    img = x[sample_idx].squeeze().cpu().numpy()  # (D, H, W)
    mask = y[sample_idx].squeeze().cpu().numpy()  # (D, H, W) 或 (D, H, W, 2)
    
    D = img.shape[0]
    step = max(1, D // max_slices)
    slices = range(0, D, step)
    n_slices = len(slices)
    
    fig, axes = plt.subplots(2, n_slices, figsize=(3*n_slices, 6))
    
    for i, s in enumerate(slices):
        axes[0, i].imshow(img[s], cmap='gray')
        axes[0, i].set_title(f"Slice {s}")
        axes[0, i].axis('off')
        
        if len(mask.shape) == 3: 
            axes[1, i].imshow(mask[s], cmap='gray')
        else: 
            axes[1, i].imshow(mask[s, :, :, 0], cmap='gray')
        
        axes[1, i].set_title(f"Mask {s}")
        axes[1, i].axis('off')
    
    plt.suptitle(f"Sample {sample_idx}")
    plt.tight_layout()
    plt.show()

In [11]:
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange, repeat
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.losses import DiceLoss
from typing import List, Optional
   
class SkeletonRecallPlusDiceLoss(nn.Module):
    def __init__(self, num_classes=3, w_srec=0.2, w_fp=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.w_srec = w_srec
        self.w_fp = w_fp
        
        self.base_loss = DiceCELoss(include_background=False,
                                to_onehot_y = True,
                                sigmoid = False,
                                softmax = False,
                                other_act = None,
                                squared_pred = False,
                                jaccard = False,
                                reduction = 'mean',
                                smooth_nr = 1e-05,
                                smooth_dr = 1e-05,
                                batch = True,
                                weight = None,
                                lambda_dice = 1.0,
                                lambda_ce = 1.0,
                                label_smoothing = 0.05,)

    
    def forward(self, y_pred, y_true):
        """
        y_pred: (B, C, D, H, W)
        y_true: (B, 2, D, H, W) [mask, skeleton]
        """
        y_true_mask = y_true[:, 0:1]  # (B, 1, D, H, W)
        y_true_skel = y_true[:, 1:2]  # (B, 1, D, H, W)

        valid_mask = y_true_mask != 0

        y_pred = F.softmax(y_pred, dim=1)
        
        pred_ink = y_pred[:, 2:3]

        
        base_loss = self.base_loss(y_pred*valid_mask, y_true_mask*valid_mask)


        
        # intersection = torch.sum(pred_ink * y_true_skel * valid_mask, dim=[2, 3, 4])
        # skeleton_sum = torch.sum(y_true_skel * valid_mask, dim=[2, 3, 4])
        
        # has_skeleton = (skeleton_sum > 0).float()
        # recall = (intersection + 1e-6) / (skeleton_sum + 1e-6)
        # skel_loss = torch.mean((1.0 - recall) * has_skeleton)

        recall = (pred_ink * y_true_skel).sum([-3,-2,-1]) / (y_true_skel.sum([-3,-2,-1])+1e-6)
        skel_loss = (1-recall).mean()
        
        gt_bg = (y_true_mask == 1).float()
        fp_volume = pred_ink * gt_bg * valid_mask
        fp_loss = torch.sum(fp_volume) / (torch.sum(gt_bg * valid_mask) + 1e-6)
        
        # 总损失
        total_loss = base_loss + self.w_srec * skel_loss + self.w_fp * fp_loss
        
        return total_loss, {'base':base_loss,'skel':skel_loss,'fp':fp_loss}

class SkeletonLossMonitor:
    def __init__(self):
        self.total = 0
        self.count = 0
    
    def update(self, y_pred, y_true):
        if y_true.shape[1] == 2: 
            y_true_skel = y_true[:, 1:2]
            pred_ink = y_pred[:, 2:3]
            
            intersection = torch.sum(pred_ink * y_true_skel, dim=[2, 3, 4])
            skeleton_sum = torch.sum(y_true_skel, dim=[2, 3, 4])
            
            has_skeleton = (skeleton_sum > 0).float()
            recall = (intersection + 1e-6) / (skeleton_sum + 1e-6)
            skel_loss = torch.mean((1.0 - recall) * has_skeleton)
            
            self.total += skel_loss.item()
            self.count += 1
    
    def reset(self):
        self.total = 0
        self.count = 0
    
    def compute(self):
        return self.total / (self.count + 1e-6)


def skel_loss(y_pred, y_true):
    y_true_skel = y_true[:, 1:2]
    pred_ink = y_pred[:, 2:3]
    
    intersection = torch.sum(pred_ink * y_true_skel, dim=[2, 3, 4])
    skeleton_sum = torch.sum(y_true_skel, dim=[2, 3, 4])
    
    has_skeleton = (skeleton_sum > 0).float()
    recall = (intersection + 1e-6) / (skeleton_sum + 1e-6)
    skel_loss = torch.mean((1.0 - recall) * has_skeleton)

    return skel_loss

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.conv(x)

class Encoder3D(nn.Module):
    def __init__(self, in_chans=1, base_ch=32, num_blocks=[2, 2, 2, 2]):
        super().__init__()
        self.enc1 = self._make_layer(in_chans, base_ch, num_blocks[0])  # 160->80
        self.enc2 = self._make_layer(base_ch, base_ch*2, num_blocks[1])  # 80->40
        self.enc3 = self._make_layer(base_ch*2, base_ch*4, num_blocks[2])  # 40->20
        self.enc4 = self._make_layer(base_ch*4, base_ch*8, num_blocks[3])  # 20->10
        self.pool = nn.MaxPool3d(2)
    
    def _make_layer(self, in_ch, out_ch, blocks):
        layers = [ConvBlock3D(in_ch, out_ch)]
        for _ in range(1, blocks):
            layers.append(ConvBlock3D(out_ch, out_ch))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        skips = []
        x1 = self.enc1(x); skips.append(x1)  # B,32,160,160,160
        x2 = self.pool(x1); x2 = self.enc2(x2); skips.append(x2)  # B,64,80,80,80
        x3 = self.pool(x2); x3 = self.enc3(x3); skips.append(x3)  # B,128,40,40,40
        x4 = self.pool(x3); x4 = self.enc4(x4); skips.append(x4)  # B,256,20,20,20
        return x4, skips[::-1] 

class Decoder3D(nn.Module):
    def __init__(self, base_ch=32, num_classes=3):
        super().__init__()
        self.up1 = nn.ConvTranspose3d(base_ch*8, base_ch*4, 2, stride=2)  # 20->40
        self.dec1 = ConvBlock3D(base_ch*4 + base_ch*4, base_ch*4)
        self.up2 = nn.ConvTranspose3d(base_ch*4, base_ch*2, 2, stride=2)  # 40->80
        self.dec2 = ConvBlock3D(base_ch*2 + base_ch*2, base_ch*2)
        self.up3 = nn.ConvTranspose3d(base_ch*2, base_ch, 2, stride=2)  # 80->160
        self.dec3 = ConvBlock3D(base_ch + base_ch, base_ch)
        self.final_conv = nn.Conv3d(base_ch, num_classes, 1)
    
    def forward(self, x, skips):
        x = self.up1(x)
        x = self.dec1(torch.cat([x, skips[0]], dim=1))
        x = self.up2(x)
        x = self.dec2(torch.cat([x, skips[1]], dim=1))
        x = self.up3(x)
        x = self.dec3(torch.cat([x, skips[2]], dim=1))
        return self.final_conv(x)

class TransUNet3D(nn.Module):
    def __init__(self, in_chans=1, num_classes=3, base_ch=32, embed_dim=768, depth=8, num_heads=12, patch_size=4):
        super().__init__()
        self.base_ch = base_ch
        self.in_chans = in_chans
        
        self.encoder = Encoder3D(in_chans, base_ch=base_ch)
        self.final_ch = base_ch * 8
        
        self.patch_embed = PatchEmbed3D(self.final_ch, embed_dim, patch_size)
        self.transformer = TransformerEncoder(embed_dim, depth, num_heads)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.decoder = Decoder3D(base_ch=base_ch, num_classes=num_classes)
        self.norm = nn.LayerNorm(embed_dim)
        self.patch_size = patch_size

    def forward(self, x):
        feat, skips = self.encoder(x)  # feat: B, final_ch, 20,20,20
        B = feat.shape[0]
        
        x = self.patch_embed(feat)
        
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = self.transformer(x)
        x = self.norm(x[:, 1:])  # B,125,768
        
        Dp = Hp = Wp = feat.shape[2] // self.patch_size 
        x = x.transpose(1, 2).reshape(B, -1, Dp, Hp, Wp)  # B,768,5,5,5
        x = F.interpolate(x, size=feat.shape[2:], mode='trilinear', align_corners=False)
        
        proj = nn.Conv3d(self.patch_embed.proj.out_channels, self.final_ch, 1).to(x.device)
        x = proj(x)  # 768 -> 1280 (for base_ch=160)
        
        out = self.decoder(x, skips)
        return out


In [13]:
class PatchEmbed3D(nn.Module):
    def __init__(self, in_chans, embed_dim, patch_size):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv3d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        B, C, D, H, W = x.shape
        x = self.proj(x)  # B, embed_dim, D//p, H//p, W//p
        x = x.flatten(2).transpose(1, 2)  # B, N, embed_dim
        return x

# 2. Transformer Block
class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads=8, mlp_dim=2048, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        # Self-attention with residual
        attn_out, _ = self.attn(self.norm1(x), self.norm1(x), self.norm1(x))
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x

class TransformerEncoder(nn.Module):
    def __init__(self, embed_dim=512, depth=8, num_heads=8, mlp_dim=2048, dropout=0.1):
        super().__init__()
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_dim, dropout)
            for _ in range(depth)
        ])
    
    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        return x

class ConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.conv(x)

class Encoder3D(nn.Module):
    def __init__(self, in_chans=1, base_ch=32, num_blocks=[2, 2, 2, 2]):
        super().__init__()
        self.enc1 = self._make_layer(in_chans, base_ch, num_blocks[0])
        self.enc2 = self._make_layer(base_ch, base_ch*2, num_blocks[1])
        self.enc3 = self._make_layer(base_ch*2, base_ch*4, num_blocks[2])
        self.enc4 = self._make_layer(base_ch*4, base_ch*8, num_blocks[3])
    
    def _make_layer(self, in_ch, out_ch, blocks):
        layers = [ConvBlock3D(in_ch, out_ch)]
        for _ in range(1, blocks):
            layers.append(ConvBlock3D(out_ch, out_ch))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        skips = []
        x1 = self.enc1(x); skips.append(x1)
        x = F.max_pool3d(x1, 2) 
        
        x2 = self.enc2(x); skips.append(x2)
        x = F.max_pool3d(x2, 2) 
        
        x3 = self.enc3(x); skips.append(x3)
        x = F.max_pool3d(x3, 2)
        
        x4 = self.enc4(x); skips.append(x4)
        
        return x4, skips[::-1] 


class Decoder3D(nn.Module):
    def __init__(self, base_ch=32, num_classes=3):
        super().__init__()
        self.base_ch = base_ch
        
        self.up1 = nn.ConvTranspose3d(base_ch*8, base_ch*4, 2, stride=2) 
        self.dec1 = ConvBlock3D(base_ch*4 + base_ch*8, base_ch*4) 
        self.up2 = nn.ConvTranspose3d(base_ch*4, base_ch*2, 2, stride=2)  
        self.dec2 = ConvBlock3D(base_ch*2 + base_ch*4, base_ch*2)
        self.up3 = nn.ConvTranspose3d(base_ch*2, base_ch, 2, stride=2) 
        self.dec3 = ConvBlock3D(base_ch + base_ch*2, base_ch)   
        self.final = nn.Conv3d(base_ch, num_classes, 1)
    
    def forward(self, x, skips):
        x = self.up1(x)         
        x = F.interpolate(x, skips[0].shape[2:])
        x = self.dec1(torch.cat([x, skips[0]], dim=1)) 
        
        x = self.up2(x)       
        x = self.dec2(torch.cat([x, skips[1]], dim=1)) 
        
        x = self.up3(x)  
        x = self.dec3(torch.cat([x, skips[2]], dim=1)) 
        
        x = F.interpolate(x, (160, 160, 160), mode='trilinear')
        return self.final(x)



class TransUNet3D(nn.Module):
    def __init__(self, in_chans=1, num_classes=3, base_ch=32, embed_dim=512, depth=8, num_heads=8, patch_size=4):
        super().__init__()
        self.base_ch = base_ch
        self.final_ch = base_ch * 8
        self.patch_size = patch_size
        
        self.encoder = Encoder3D(in_chans, base_ch)
        self.patch_embed = PatchEmbed3D(self.final_ch, embed_dim, patch_size)
        self.transformer = TransformerEncoder(embed_dim, depth, num_heads)  # ✅ DEFINED
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.decoder = Decoder3D(base_ch, num_classes)
        self.norm = nn.LayerNorm(embed_dim)
    
    def forward(self, x):
        feat, skips = self.encoder(x)
        
        B = feat.shape[0]
        x = self.patch_embed(feat)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = self.transformer(x)
        x = self.norm(x[:, 1:])
        
        # Reshape & project
        Dp = feat.shape[2] // self.patch_size
        x = x.transpose(1, 2).reshape(B, -1, Dp, Dp, Dp)
        x = F.interpolate(x, size=feat.shape[2:], mode='trilinear')
        proj = nn.Conv3d(self.patch_embed.proj.out_channels, self.final_ch, 1).to(x.device)
        x = proj(x)
        
        out = self.decoder(x, skips)
        return out


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Сначала переносим модель на GPU
# model = UNETR(
#     in_channels=1,
#     out_channels=3,
#     img_size=(160,160,160),
#     feature_size = 32,
#     hidden_size = 500,
#     mlp_dim = 1512,
#     num_heads = 10,
#     proj_type = 'conv',
#     norm_name = 'instance',
#     conv_block = True,
#     res_block = True,
#     dropout_rate = 0.2,
#     spatial_dims = 3,
#     qkv_bias = False,
#     save_attn = False,
# ).to(device)


model = TransUNet3D(
    in_chans=1, num_classes=3, base_ch=32, embed_dim=768, depth=8, num_heads=12, patch_size=4
).to(device)

# model = TransUNet3D(in_chans=1, num_classes=3, base_ch=48, embed_dim=500, depth=10, 
#                  num_heads=10, patch_size=4, mlp_dim=2048, dropout=0.1).to(device)

# 3. Только потом оборачиваем в DataParallel
if torch.cuda.device_count() > 1:
    print(f"Using DataParallel with {torch.cuda.device_count()} GPUs")
    model = torch.nn.DataParallel(model)


Using DataParallel with 2 GPUs


In [15]:
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 67,584,099


In [16]:
from monai.metrics import DiceMetric

In [17]:
total_steps = (num_samples // batch_size) * epochs
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=total_steps,
    eta_min=5e-7
)

loss_fn = SkeletonRecallPlusDiceLoss(num_classes=num_classes, w_srec=0.05, w_fp=0.05)

train_dice_metric = DiceMetric(
    include_background=False,      
    reduction="mean",              # Average across batch
    get_not_nans=True
)

val_dice_metric = DiceMetric(
    include_background=False,
    reduction="mean_batch",        # Accumulate then average
    get_not_nans=True
)

skel_monitor = SkeletonLossMonitor()

class SlidingWindowInferenceCallback:
    def __init__(self, model, val_loader, roi_size, sw_batch_size=4, overlap=0.5, interval=10):
        self.model = model
        self.val_loader = val_loader
        self.roi_size = roi_size
        self.sw_batch_size = sw_batch_size
        self.overlap = overlap
        self.interval = interval
        self.inferer = SlidingWindowInferer(
            roi_size=roi_size,
            sw_batch_size=sw_batch_size,
            overlap=overlap,
            mode="gaussian"
        )
    
    def __call__(self, epoch, metrics):
        if (epoch + 1) % self.interval == 0:
            self.model.eval()
            dice_values = []
            
            with torch.no_grad():
                for batch in self.val_loader:
                    images, labels = batch
                    images = images.to(device)
                    labels = labels.to(device)
                    
                    outputs = self.inferer(images, self.model)
                    
                    dice_metric(outputs.argmax(dim=1, keepdim=True), labels)
                    dice = dice_metric.aggregate().item()
                    dice_values.append(dice)
                    dice_metric.reset()
            
            avg_dice = np.mean(dice_values) if dice_values else 0
            print(f"\n[SWI] Epoch {epoch+1}, Validation Dice: {avg_dice:.4f}")
            
            if avg_dice > self.best_dice:
                self.best_dice = avg_dice
                torch.save(self.model.state_dict(), f"best_model_epoch_{epoch+1}.pth")
            
            self.model.train()

In [18]:
class PeriodicWeightsSaver:
    def __init__(self, interval=25, save_path_template="checkpoint_epoch_{}.pth"):
        self.interval = interval
        self.save_path_template = save_path_template
    
    def __call__(self, epoch, model):
        if (epoch + 1) % self.interval == 0:
            save_path = self.save_path_template.format(epoch + 1)
            
            if use_ddp and hasattr(model, 'module'):
                torch.save(model.module.state_dict(), save_path)
            else:
                torch.save(model.state_dict(), save_path)
            
            print(f"\n[Snapshot] Saved periodic weights to: {save_path}")

In [19]:
from monai.losses import DiceCELoss

loss_fn = DiceCELoss(
    include_background=False,    # Ignores class 0 perfectly
    to_onehot_y=True,           # Handles your 0/1 labels → 3ch internally
    squared_pred=True,          # Stabilizes sparse ink
    smooth_nr=1e-5,
    softmax=True,
    smooth_dr=1e-5
)

In [20]:
def train_epoch(model, loader, optimizer, loss_fn, epoch, scaler=None):
    """训练一个epoch"""
    model.train()
    total_loss = 0
    dice_scores = []
    
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}")
    for batch_idx, (images, labels) in enumerate(pbar):
        images = images.to(device)
        labels = labels.to(device).long()[:,:1]
        optimizer.zero_grad()

        if scaler:
            with torch.cuda.amp.autocast(enabled=True, dtype=torch.bfloat16):
                outputs = model(images)
                loss = loss_fn(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        
        pred_argmax = outputs.detach().cpu().argmax(dim=1, keepdim=True)  # (B,1,D,H,W)
        train_dice_metric(y_pred=pred_argmax, y=labels.detach().cpu().unsqueeze(1))
        

        dice = train_dice_metric.aggregate()[0]
        
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            # 'base': f'{loss_components['base']:.4f}',
            # 'skel': f'{loss_components['skel']:.4f}',
            # 'fp': f'{loss_components['fp']:.4f}',
            'Dice': f'{dice.item():.4f}',

        })
        
        scheduler.step()
    
    avg_loss = total_loss / len(loader)
    
    return avg_loss

def validate(model, loader, loss_fn):
    """验证"""
    model.eval()
    total_loss = 0
    dice_values = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validation"):
            images = images.to(device)
            labels = labels.to(device).long()
            
            outputs = model(images)

            loss = loss_fn(outputs, labels)
            
            total_loss += loss.item()
            
    avg_loss = total_loss / len(loader)
    
    return avg_loss

def main():
    swi_callback = SlidingWindowInferenceCallback(
        model, val_loader, input_shape, interval=10
    )
    snapshot_cb = PeriodicWeightsSaver(interval=40)

    scaler = torch.cuda.amp.GradScaler(enabled=True) if torch.cuda.is_available() else None

    train_history = {'loss': []}
    val_history = {'loss': []}
    
    best_val_loss = 0
    
    for epoch in range(epochs):
        if use_ddp:
            train_loader.sampler.set_epoch(epoch)

        train_loss = train_epoch(
            model, train_loader, optimizer, loss_fn, epoch, scaler
        )
        train_history['loss'].append(train_loss)

        if rank == 0:
            val_loss = validate(model, val_loader, loss_fn)
            val_history['loss'].append(val_loss)
            
            print(f"\nEpoch {epoch+1}/{epochs}")
            print(f"Train Loss: {train_loss:.4f}")
            print(f"Val Loss: {val_loss:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), "best_model.pth")
                print(f"Saved best model with Dice: {best_val_loss:.4f}")
        
        if rank == 0:
            snapshot_cb(epoch, model)
            
    if rank == 0:
        torch.save(model.state_dict(), "final_model.pth")

        plt.figure(figsize=(12, 4))
        plt.subplot(1, 2, 1)
        plt.plot(train_history['loss'], label='Train Loss')
        plt.plot(val_history['loss'], label='Val Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.title('Training and Validation Loss')
        
        plt.tight_layout()
        plt.savefig('training_history.png')
        plt.show()

In [ ]:
if __name__ == "__main__":
    if rank == 0:
        print("Starting training...")
        print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
        print(f"Batch size: {batch_size}")
        print(f"Epochs: {epochs}")
    
    main()
    
    if use_ddp:
        dist.destroy_process_group()

/tmp/ipykernel_3709/2949314902.py:84: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=True) if torch.cuda.is_available() else None


Starting training...
Model parameters: 67,584,099
Batch size: 2
Epochs: 300


Epoch 1:   0%|          | 0/353 [00:00<?, ?it/s]/tmp/ipykernel_3709/2949314902.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True, dtype=torch.bfloat16):
Epoch 1:  90%|████████▉ | 316/353 [24:38<02:53,  4.70s/it, loss=1.4197, Dice=0.4512]